In [19]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
from scipy.stats import qmc
from heston_project.pricing.COS_with_jax_gradients import *

In [20]:
df_final = pd.read_csv('../../data/dataset_pricing_and_greeks_200000(1).csv')

# Identifiera negativa priser för loggning
neg_mask = df_final["price"] < 0
print(f"Antal rader med negativa priser som tas bort: {neg_mask.sum()} / {len(df_final)}")

# Behåll endast rader där priset är 0.0 eller högre
df_final = df_final[df_final["price"] >= 0].copy()

# Återställ index så att vi inte har luckor från de borttagna raderna
df_final.reset_index(drop=True, inplace=True)

print(f"Antal rader kvar: {len(df_final)}")
print(f"Minsta pris nu: {df_final['price'].min()}")

Antal rader med negativa priser som tas bort: 171 / 200000
Antal rader kvar: 199829
Minsta pris nu: 0.0


In [21]:
# Check if prices satisfy intrinsic value bound: p >= max(K*e^(-rT) - S, 0)

S = df_final["S"].values
K = df_final["K"].values
T = df_final["T"].values
r = df_final["r"].values

intrinsic = np.maximum(K * np.exp(-r * T) - S, 0.0)
prices = df_final["price"].values

#valid = (prices >= intrinsic - 1e-10).sum()
valid = (prices >= intrinsic).sum()
print(f"Priser som uppfyller p >= max(K*e^(-rT) - S, 0): {valid} / {len(prices)}")

Priser som uppfyller p >= max(K*e^(-rT) - S, 0): 199285 / 199829


In [22]:
# Calculate K*e^(rT) - S and show only violations
K_forward = K * np.exp(r * T)
forward_moneyness = K_forward - S

# Find rows that violate p >= max(K*e^(-rT) - S, 0)
violation_mask = (prices < intrinsic) & (prices >= intrinsic - 1e-14)
large_violation_mask = prices < intrinsic - 1e-14
violated = df_final[violation_mask].copy()
violated["K*e^(rT)"] = K_forward[violation_mask]
violated["K*e^(rT) - S"] = forward_moneyness[violation_mask]

print(f"Antal priser som INTE uppfyller p >= max(K*e^(-rT) - S, 0): {violation_mask.sum()} / {len(prices)}")
print("="*150)

print(f"Antal priser som INTE uppfyller p >= max(K*e^(-rT) - S, 0) - esp: {large_violation_mask.sum()} / {len(prices)}")

if violation_mask.sum() > 0:
    params_to_show = ["K*e^(rT) - S", "price", "S", "K", "kappa", "theta", "sigma", "rho", "v0", "T", "r", "lm"]
    available_cols = [col for col in params_to_show if col in violated.columns]
    print(violated[available_cols].to_string())
else:
    print("Inga priser som INTE uppfyller p >= max(K*e^(-rT) - S, 0) hittades.")

Antal priser som INTE uppfyller p >= max(K*e^(-rT) - S, 0): 528 / 199829
Antal priser som INTE uppfyller p >= max(K*e^(-rT) - S, 0) - esp: 16 / 199829
        K*e^(rT) - S     price    S         K     kappa     theta     sigma       rho        v0         T         r        lm
458         2.579769  2.579480  1.0  3.579624  1.222589  0.427757  1.200439 -0.870931  0.213030  0.038570  0.001046 -1.275258
865         2.379655  2.372038  1.0  3.375844  2.912909  0.610665  0.970135 -0.232536  0.712780  0.029455  0.038302 -1.216645
1224        2.214798  2.212402  1.0  3.213600  3.942872  0.138254  0.636947 -0.632054  0.528005  0.043459  0.008577 -1.167392
1403        1.089689  1.089373  1.0  2.089531  0.176421  0.625235  0.700844 -0.496052  0.034108  0.024277  0.003117 -0.736940
1536        3.068397  2.961175  1.0  4.014428  4.724096  0.073352  1.209949 -0.902269  0.018897  0.294684  0.045317 -1.389895
2478        3.312228  3.270322  1.0  4.291224  2.411967  0.456882  0.953002 -0.876679  0.6336

In [23]:
tolerance = 1e-14
near_intrinsic_mask = (prices < intrinsic) & (intrinsic - prices <= tolerance)

print(f"### Justering för priser nära intrinsic (endast under) ###")
print(f"Antal priser mellan 0 och {tolerance} under intrinsic: {near_intrinsic_mask.sum()}")

if near_intrinsic_mask.sum() > 0:
    violations_under = intrinsic[near_intrinsic_mask] - prices[near_intrinsic_mask]
    print(f"Min gap: {np.min(violations_under):.2e}")
    print(f"Max gap: {np.max(violations_under):.2e}")
    print(f"Medel gap: {np.mean(violations_under):.2e}")
    
    # Justera dessa priser till intrinsic value
    df_final.loc[near_intrinsic_mask, "price"] = intrinsic[near_intrinsic_mask]
    prices = df_final["price"].values
    
    print(f"✓ {near_intrinsic_mask.sum()} priser justerade till intrinsic value")
else:
    print("Inga priser som är nära intrinsic hittades.")

### Justering för priser nära intrinsic (endast under) ###
Antal priser mellan 0 och 1e-14 under intrinsic: 528
Min gap: 1.11e-16
Max gap: 8.88e-15
Medel gap: 1.48e-15
✓ 528 priser justerade till intrinsic value


In [24]:
# Set prices that are MORE than 1e-14 BELOW intrinsic to NaN
tolerance = 1e-14
large_violation_mask = (prices < intrinsic) & (intrinsic - prices > tolerance)

print(f"### Sätt större överträdelser till NaN ###")
print(f"Antal priser mer än {tolerance} under intrinsic: {large_violation_mask.sum()}")

if large_violation_mask.sum() > 0:
    violations_large = intrinsic[large_violation_mask] - prices[large_violation_mask]
    print(f"Min gap: {np.min(violations_large):.2e}")
    print(f"Max gap: {np.max(violations_large):.2e}")
    print(f"Medel gap: {np.mean(violations_large):.2e}")
    
    # Sätt dessa priser till NaN
    df_final.loc[large_violation_mask, "price"] = np.nan
    prices = df_final["price"].values
    
    print(f"✓ {large_violation_mask.sum()} priser satta till NaN")

### Sätt större överträdelser till NaN ###
Antal priser mer än 1e-14 under intrinsic: 16
Min gap: 1.24e-14
Max gap: 1.06e-05
Medel gap: 9.87e-07
✓ 16 priser satta till NaN


In [ ]:
# Set prices that are MORE than 1e-14 ABOVE upper bound to NaN
# Upper bound for put: P <= K*e^(-rT)
tolerance = 1e-14
upper_bound = K * np.exp(-r * T)
upper_violation_mask = (prices > upper_bound) & (prices - upper_bound > tolerance)

print(f"\n### Sätt överträdelser över upper bound till NaN ###")
print(f"Upper bound (K*e^(-rT)) check")
print(f"Antal priser mer än {tolerance} över upper bound: {upper_violation_mask.sum()}")

if upper_violation_mask.sum() > 0:
    violations_upper = prices[upper_violation_mask] - upper_bound[upper_violation_mask]
    print(f"Min gap över upper bound: {np.min(violations_upper):.2e}")
    print(f"Max gap över upper bound: {np.max(violations_upper):.2e}")
    print(f"Medel gap över upper bound: {np.mean(violations_upper):.2e}")
    
    # Sätt dessa priser till NaN
    df_final.loc[upper_violation_mask, "price"] = np.nan
    prices = df_final["price"].values
    
    print(f"✓ {upper_violation_mask.sum()} priser satta till NaN")
else:
    print("Inga priser behövde sättas till NaN för upper bound")


### Sätt överträdelser över upper bound till NaN ###
Upper bound (K*e^(-rT)) check
Antal priser mer än 1e-14 över upper bound: 0
Inga priser behövde sättas till NaN för upper bound


In [ ]:
from pathlib import Path

out_dir = Path("../../data")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"dataset_arbitrage_fix_200000(1).csv"
df_final.to_csv(out_path, index=False)

print(f"Sparade {len(df_final)} rader till {out_path.resolve()}")
print(f"Kolumner: {list(df_final.columns)}")